In [1]:
import json
import bluequbit
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFT, QFTGate

bq = bluequbit.init("lEiTmm6zeLxxZ6q3aKBMsxwhrdnDr7vF")

[BQ-PYTHON-SDK][WARNING] - Beta version 0.18.1b1 of BlueQubit Python SDK is being used.


In [19]:
from qiskit.synthesis import synth_qft_full

options = {
    "mps_bond_dimension": 256,
}
num_qubits = 96
qft_circuit = synth_qft_full(num_qubits=num_qubits)
qft_circuit.draw('mpl')
num_gates = qft_circuit.size()
depth = qft_circuit.depth()
job = bq.run(qft_circuit, device="mps.cpu", options=options)
queue_time = job.queue_time_ms
run_time = job.run_time_ms
run_time_per_gate = run_time / num_gates if num_gates > 0 else 0
print(f"Depth: {depth}, Number of Gates: {num_gates}, Runtime: {run_time}, Queuetime: {queue_time}, Run time per gate (ms): {run_time_per_gate:.3f}")

[BQ-PYTHON-SDK][INFO] - Submitted: Job ID: BpP0X7LT3P5Wofjm, device: mps.cpu, run status: RUNNING, created on: 2026-01-19 21:36:55 UTC, estimated runtime: 360000000 ms, estimated cost: $30.00, num qubits: 96, shots: 1024
Depth: 192, Number of Gates: 4704, Runtime: 35855, Queuetime: 221, Run time per gate (ms): 7.622


In [20]:
from qiskit.circuit.library import quantum_volume

In [ ]:
options = {
    "mps_bond_dimension": 32,
}
num_qubits = 96
depth = 32
qc = quantum_volume(num_qubits, depth, seed=42)
num_gates = qc.size()
depth = qc.depth()
job = bq.run(qc, device="mps.cpu", options=options)
queue_time = job.queue_time_ms
run_time = job.run_time_ms
run_time_per_gate = run_time / num_gates if num_gates > 0 else 0
print(f"Depth: {depth}, Number of Gates: {num_gates}, Runtime: {run_time}, Queuetime: {queue_time}, Run time per gate (ms): {run_time_per_gate:.3f}")

In [ ]:
def compute_two_q_depth(qc):
    """
    Compute the maximum number of 2-qubit gates any single qubit is involved in.
    """
    from collections import defaultdict
    qubit_2q_counts = defaultdict(int)
    
    for instruction in qc.data:
        qubits = instruction.qubits
        if len(qubits) == 2:
            for qubit in qubits:
                qubit_2q_counts[qubit] += 1
    
    return max(qubit_2q_counts.values()) if qubit_2q_counts else 0

In [ ]:
num_qubits = 4
qc = QuantumCircuit(num_qubits)
qc.h(0)
for i in range(num_qubits - 1):
     qc.cx(i, i+1)
print(qc.size())
print(compute_two_q_depth(qc))

40


In [ ]:
# Test MPS on CPU with small bond dimension
print("Testing MPS on CPU...")
result_cpu = bq.run(qft_circuit, device="mps.cpu", bond_dimension=2)
print(f"CPU result: {result_cpu}")
print(f"CPU run_time_ms: {result_cpu.run_time_ms}")

In [ ]:
# Test MPS on GPU with small bond dimension
print("Testing MPS on GPU...")
result_gpu = bq.run(qft_circuit, device="mps.gpu", bond_dimension=2)
print(f"GPU result: {result_gpu}")
print(f"GPU run_time_ms: {result_gpu.run_time_ms}")

In [ ]:
# Scaling experiment: gradually increase bond dimension and qubit count
import time

configs = [
    {"num_qubits": 4, "bond_dim": 2, "device": "mps.cpu"},
    {"num_qubits": 4, "bond_dim": 4, "device": "mps.cpu"},
    {"num_qubits": 8, "bond_dim": 2, "device": "mps.cpu"},
    {"num_qubits": 8, "bond_dim": 4, "device": "mps.cpu"},
]

results = []
for config in configs:
    print(f"\nTesting: {config}")
    qft = synth_qft_full(num_qubits=config["num_qubits"])
    print(f"  Circuit depth: {qft.depth()}, gates: {qft.size()}")
    
    try:
        result = bq.run(qft, device=config["device"], bond_dimension=config["bond_dim"])
        results.append({**config, "run_time_ms": result.run_time_ms, "status": "success"})
        print(f"  ✓ {result.run_time_ms}ms")
    except Exception as e:
        results.append({**config, "error": str(e), "status": "failed"})
        print(f"  ✗ {e}")

print("\n" + "="*50)
print("Summary:")
for r in results:
    if r["status"] == "success":
        print(f"  {r['num_qubits]} qubits, bond_dim={r['bond_dim']}, {r['device']}: {r['run_time_ms']}ms")
    else:
        print(f"  {r['num_qubits']} qubits, bond_dim={r['bond_dim']}, {r['device']}: FAILED - {r['error']}")